<a href="https://colab.research.google.com/github/MhiztaDhee/Machine-Learning-for-Agricultural-Prices-in-Nigeria/blob/main/Corrected_Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Import necessary libraries
import pandas as pd

#Importing dataset
from google.colab import files
uploaded = files.upload()



Saving wfp_food_prices_nga (1).csv to wfp_food_prices_nga (1).csv


In [ ]:
#Load dataset into dataframe
df = pd.read_csv("wfp_food_prices_nga (1).csv")

#Preview first 5 rows
df.head()

/tmp/ipykernel_2208/238088096.py:2: DtypeWarning: Columns (4,5,6,9,14,15) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("wfp_food_prices_nga (1).csv")


,date,admin1,admin2,market,market_id,latitude,longitude,category,commodity,commodity_id,unit,priceflag,pricetype,currency,price,usdprice
0,#date,#adm1+name,#adm2+name,#loc+market+name,#loc+market+code,#geo+lat,#geo+lon,#item+type,#item+name,#item+code,#item+unit,#item+price+flag,#item+price+type,#currency+code,#value,#value+usd
1,2002-01-15,Katsina,Jibia,Jibia (CBM),1038,13.08,7.24,cereals and tubers,Maize,51,KG,actual,Wholesale,NGN,175.92,1.54
2,2002-01-15,Katsina,Jibia,Jibia (CBM),1038,13.08,7.24,cereals and tubers,Millet,73,KG,actual,Wholesale,NGN,150.18,1.31
3,2002-01-15,Katsina,Jibia,Jibia (CBM),1038,13.08,7.24,cereals and tubers,Rice (imported),64,KG,actual,Wholesale,NGN,358.7,3.14
4,2002-01-15,Katsina,Jibia,Jibia (CBM),1038,13.08,7.24,cereals and tubers,Sorghum,65,KG,actual,Wholesale,NGN,155.61,1.36


In [ ]:
# Convert columns
df['price'] = pd.to_numeric(df['price'], errors='coerce')
df['usdprice'] = pd.to_numeric(df['usdprice'], errors='coerce')

df['date'] = pd.to_datetime(df['date'], errors='coerce')

# Drop missing values
df = df.dropna(subset=['price', 'usdprice', 'date'])

# Check
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 60565 entries, 1 to 60565
Data columns (total 16 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   date          60565 non-null  datetime64[ns]
 1   admin1        60565 non-null  object        
 2   admin2        60565 non-null  object        
 3   market        60565 non-null  object        
 4   market_id     60565 non-null  object        
 5   latitude      60565 non-null  object        
 6   longitude     60565 non-null  object        
 7   category      60565 non-null  object        
 8   commodity     60565 non-null  object        
 9   commodity_id  60565 non-null  object        
 10  unit          60565 non-null  object        
 11  priceflag     60565 non-null  object        
 12  pricetype     60565 non-null  object        
 13  currency      60565 non-null  object        
 14  price         60565 non-null  float64       
 15  usdprice      60565 non-null  float64    

/tmp/ipykernel_2208/3429925478.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['date'] = pd.to_datetime(df['date'], errors='coerce')


In [ ]:
# Extract time features
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month

# Check
df[['date', 'year', 'month']].head()

,date,year,month
1,2002-01-15,2002,1
2,2002-01-15,2002,1
3,2002-01-15,2002,1
4,2002-01-15,2002,1
5,2002-01-15,2002,1


In [ ]:
# Create lag feature (previous price)
df['lag_1'] = df.groupby(['commodity', 'admin1'])['price'].shift(1)

# Drop missing values created by lag
df = df.dropna()

# Check
df[['price', 'lag_1']].head()

,price,lag_1
9,169.76,175.92
10,381.97,358.70
12,181.94,169.76
13,175.00,150.18
14,361.11,381.97


In [ ]:
# More lag features
df['lag_2'] = df.groupby(['commodity', 'admin1'])['price'].shift(2)
df['lag_3'] = df.groupby(['commodity', 'admin1'])['price'].shift(3)

# Drop missing again
df = df.dropna()

# Check
df[['price', 'lag_1', 'lag_2', 'lag_3']].head()

,price,lag_1,lag_2,lag_3
27,208.67,190.22,181.94,169.76
36,222.26,208.67,190.22,181.94
37,216.97,195.12,187.50,175.00
39,187.82,189.04,188.37,171.25
40,343.93,353.53,357.34,345.36


In [ ]:
# Price change (trend)
df['price_change'] = df['price'] - df['lag_1']

# Check
df[['price', 'lag_1', 'price_change']].head()

,price,lag_1,price_change
27,208.67,190.22,18.45
36,222.26,208.67,13.59
37,216.97,195.12,21.85
39,187.82,189.04,-1.22
40,343.93,353.53,-9.60


In [ ]:
# Define features
features = [
    'year', 'month',
    'admin1', 'commodity',
    'lag_1', 'lag_2', 'lag_3'
]

df['target_price'] = df.groupby(['commodity','admin1'])['price'].shift(-1)
df = df.dropna()

target = 'target_price'

# Split into X and y
X = df[features]
y = df[target]

# Encode categorical variables
X = pd.get_dummies(X, drop_first=True)

# Check
X.head()

,year,month,lag_1,lag_2,lag_3,admin1_Adamawa,admin1_Borno,admin1_Gombe,admin1_Jigawa,admin1_Kaduna,...,commodity_Sorghum,commodity_Sorghum (brown),commodity_Sorghum (white),commodity_Spinach,commodity_Sugar,commodity_Tomatoes,commodity_Watermelons,commodity_Wheat,commodity_Yam,commodity_Yam (Abuja)
86,2002,12,98.75,98.33,112.50,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
90,2002,12,110.01,98.75,98.33,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
91,2002,12,125.92,120.00,130.00,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
97,2003,1,114.23,110.01,98.75,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
98,2003,1,125.28,125.92,120.00,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [ ]:
#Time-based Split

split_date = '2022-01-01'

train = df[df['date'] < split_date]
test = df[df['date'] >= split_date]

# Split X and y again using train/test
X_train = train[features]
y_train = train[target]

X_test = test[features]
y_test = test[target]

# Encode separately
X_train = pd.get_dummies(X_train, drop_first=True)
X_test = pd.get_dummies(X_test, drop_first=True)

# Align columns
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

# Check sizes
print(X_train.shape, X_test.shape)

(39894, 58) (16781, 58)


In [ ]:
print(train['date'].min(), train['date'].max())
print(test['date'].min(), test['date'].max())

2002-12-15 00:00:00 2021-12-15 00:00:00
2022-01-15 00:00:00 2025-05-15 00:00:00


In [ ]:
from sklearn.linear_model import LinearRegression

# Train model
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Predict
y_pred_lr = lr_model.predict(X_test)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae_lr = mean_absolute_error(y_test, y_pred_lr)
mse_lr = mean_squared_error(y_test, y_pred_lr)
r2_lr = r2_score(y_test, y_pred_lr)

print("Linear Regression Results:")
print("MAE:", mae_lr)
print("MSE:", mse_lr)
print("R²:", r2_lr)

Linear Regression Results:
MAE: 3687.6420767137765
MSE: 75236811.97653954
R²: 0.465106052663101


In [ ]:
print(df['price'].describe())

count     56675.000000
mean       5076.626692
std       10642.446344
min           5.000000
25%         212.960000
50%         500.000000
75%        3121.635000
max      180000.000000
Name: price, dtype: float64


In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Train model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Predict
y_pred_rf = rf_model.predict(X_test)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae_rf = mean_absolute_error(y_test, y_pred_rf)
mse_rf = mean_squared_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print("Random Forest Results:")
print("MAE:", mae_rf)
print("MSE:", mse_rf)
print("R²:", r2_rf)

Random Forest Results:
MAE: 3945.1085767817676
MSE: 82450422.9325866
R²: 0.41382109337966033


In [ ]:
# Rolling average (last 3 observations)
df['rolling_mean_3'] = df.groupby(['commodity', 'admin1'])['price'] \
                        .transform(lambda x: x.shift(1).rolling(3).mean())

# Drop missing values again
df = df.dropna()

# Check
df[['price', 'lag_1', 'rolling_mean_3']].head()

,price,lag_1,rolling_mean_3
101,114.01,109.05,111.096667
113,103.86,114.01,112.430000
114,129.65,136.06,127.230000
122,99.12,103.86,108.973333
123,116.89,129.65,128.686667


In [ ]:
# Updated feature list
features = [
    'year', 'month',
    'admin1', 'commodity',
    'lag_1', 'lag_2', 'lag_3',
    'price_change',
    'rolling_mean_3'
]

In [ ]:
# Recreate X and y
X = df[features]
y = df[target]

# Time-based split again
split_date = '2022-01-01'

train = df[df['date'] < split_date]
test = df[df['date'] >= split_date]

X_train = train[features]
y_train = train[target]

X_test = test[features]
y_test = test[target]

# Encode categorical variables
X_train = pd.get_dummies(X_train, drop_first=True)
X_test = pd.get_dummies(X_test, drop_first=True)

# Align columns
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

# Check
print(X_train.shape, X_test.shape)

(39060, 60) (16732, 60)


In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Train again
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Predict
y_pred_rf = rf_model.predict(X_test)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae_rf = mean_absolute_error(y_test, y_pred_rf)
mse_rf = mean_squared_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print("Random Forest (Improved) Results:")
print("MAE:", mae_rf)
print("MSE:", mse_rf)
print("R²:", r2_rf)

Random Forest (Improved) Results:
MAE: 3474.729378848221
MSE: 70883256.95688626
R²: 0.4974151188727709


In [ ]:
#Training ARIMA
# Example: Maize in Katsina
subset = df[(df['commodity'] == 'Maize') & (df['admin1'] == 'Katsina')]

# Sort by date
subset = subset.sort_values('date')

# Use only price
series = subset['price']

series.head()

,price
101,114.01
113,103.86
122,99.12
125,97.84
131,97.40


In [ ]:
# Split (80% train, 20% test)
train_size = int(len(series) * 0.8)

train_series = series.iloc[:train_size]
test_series = series.iloc[train_size:]

print(len(train_series), len(test_series))

181 46


In [ ]:
from statsmodels.tsa.arima.model import ARIMA

# Train ARIMA model
arima_model = ARIMA(train_series, order=(1,1,1))
arima_fit = arima_model.fit()

# Forecast
y_pred_arima = arima_fit.forecast(steps=len(test_series))

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be give

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae_arima = mean_absolute_error(test_series, y_pred_arima)
mse_arima = mean_squared_error(test_series, y_pred_arima)
r2_arima = r2_score(test_series, y_pred_arima)

print("ARIMA Results:")
print("MAE:", mae_arima)
print("MSE:", mse_arima)
print("R²:", r2_arima)

ARIMA Results:
MAE: 26.339309852505696
MSE: 1296.2915993175338
R²: -0.25890137734313634


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
import numpy as np

# Sort data (important)
df = df.sort_values('date')

# Use only a portion for demonstration (faster)
df_small = df.copy()

# Store results
r2_scores = []

# Walk-forward loop
step_size = int(len(df_small) * 0.1)

for i in range(step_size, len(df_small), step_size):

    train = df_small.iloc[:i]
    test = df_small.iloc[i:i+step_size]

    if len(test) == 0:
        break

    X_train = pd.get_dummies(train[features], drop_first=True)
    X_test = pd.get_dummies(test[features], drop_first=True)

    X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

    y_train = train[target]
    y_test = test[target]

    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    r2 = r2_score(y_test, y_pred)
    r2_scores.append(r2)

# Final result
print("Walk-forward R² scores:", r2_scores)
print("Average R²:", np.mean(r2_scores))

Walk-forward R² scores: [0.6483798246239696, 0.7867148478860989, 0.11127903933180616, 0.28025967946745933, 0.43630324059994285, 0.48012602153737105, 0.4806959473203325, 0.09070678571632118, 0.19632779459898697, -0.24280046739999994]
Average R²: 0.32679927136822884


In [ ]:
import numpy as np

# Use the same subset
data = subset['price'].values

# Normalize (VERY IMPORTANT for LSTM)
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
data_scaled = scaler.fit_transform(data.reshape(-1,1))

# Create sequences (last 3 → predict next)
X = []
y = []

sequence_length = 3

for i in range(len(data_scaled) - sequence_length):
    X.append(data_scaled[i:i+sequence_length])
    y.append(data_scaled[i+sequence_length])

X = np.array(X)
y = np.array(y)

print(X.shape, y.shape)

(224, 3, 1) (224, 1)


In [ ]:
# Split (80% train, 20% test)
train_size = int(len(X) * 0.8)

X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

print(X_train.shape, X_test.shape)

(179, 3, 1) (45, 3, 1)


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

# Build model
model = Sequential()

model.add(LSTM(50, activation='relu', input_shape=(X_train.shape[1], X_train.shape[2])))
model.add(Dense(1))

model.compile(optimizer='adam', loss='mse')

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 50)             │        10,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,451 (40.82 KB)

 Trainable params: 10,451 (40.82 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=16,
    validation_data=(X_test, y_test),
    verbose=1
)

Epoch 1/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 2s 38ms/step - loss: 0.1738 - val_loss: 0.1108
Epoch 2/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.1176 - val_loss: 0.0670
Epoch 3/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0689 - val_loss: 0.0317
Epoch 4/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0320 - val_loss: 0.0131
Epoch 5/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0170 - val_loss: 0.0113
Epoch 6/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0168 - val_loss: 0.0114
Epoch 7/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0158 - val_loss: 0.0104
Epoch 8/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0152 - val_loss: 0.0103
Epoch 9/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0151 - val_loss: 0.0103
Epoch 10/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0149 - val_loss: 0.0099
Epoch 11/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0148 - val_loss: 0.0098
Epoch 12/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0144 - v

In [ ]:
# Predict
y_pred_lstm = model.predict(X_test)

# Convert back to original scale
y_test_inv = scaler.inverse_transform(y_test)
y_pred_inv = scaler.inverse_transform(y_pred_lstm)

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae_lstm = mean_absolute_error(y_test_inv, y_pred_inv)
mse_lstm = mean_squared_error(y_test_inv, y_pred_inv)
r2_lstm = r2_score(y_test_inv, y_pred_inv)

print("LSTM Results:")
print("MAE:", mae_lstm)
print("MSE:", mse_lstm)
print("R²:", r2_lstm)

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 123ms/step
LSTM Results:
MAE: 15.764604546440976
MSE: 393.95754415548055
R²: 0.6215328486020804


In [ ]:
subset = df[(df['commodity'] == 'Rice (imported)') & (df['admin1'] == 'Katsina')]

In [ ]:
# New subset: Rice (imported) in Katsina
subset_rice = df[(df['commodity'] == 'Rice (imported)') & (df['admin1'] == 'Katsina')]

# Sort by date
subset_rice = subset_rice.sort_values('date')

# Extract price
data = subset_rice['price'].values

print(len(data))

215
